# Structured Field Extraction — LoRA Fine-Tuning (Unsloth)

Fine-tunes a small open-source model (Phi-4-mini by default) to extract
`company`, `date`, `address`, `total` as JSON from messy receipt/invoice
text, then benchmarks it against the untouched base model.

**Runs on a free Colab T4 GPU.** If you're opening this from VS Code via the
"Google Colab" extension, select the Colab kernel with a GPU runtime before
running anything below. Everything here is free — no paid API, no paid
compute.

**Before running:** push this repo to GitHub and replace `REPO_URL` in the
first code cell below with your repo's clone URL, so the notebook has
access to `data/`, `src/`, etc. on the remote Colab VM (the VM's filesystem
is separate from your laptop's).

In [ ]:
# --- Setup: install deps + pull this repo onto the Colab VM ---
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

REPO_URL = "https://github.com/varsha-sherla/field-extraction-finetune"  # <-- replace with your repo
import os
if not os.path.exists("field-extraction-finetune"):
    !git clone {REPO_URL}
%cd field-extraction-finetune

import sys
sys.path.insert(0, "src")
sys.path.insert(0, "data")


In [ ]:
# --- Sanity check: confirm a GPU is actually attached ---
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# --- If you haven't generated data yet, do one of these two: ---

# Option A (real data for your actual CV numbers -- recommended):
# !pip install -q datasets
# !python data/prepare_sroie.py

# Option B (synthetic data -- quick pipeline sanity check, not for your final numbers):
# !python data/generate_synthetic_data.py --n 150 --seed 42

import os
assert os.path.exists("data/train.jsonl") and os.path.exists("data/test.jsonl"), \
    "Run one of the two data-prep commands above first."
print("Data ready:",
      sum(1 for _ in open("data/train.jsonl")), "train examples,",
      sum(1 for _ in open("data/test.jsonl")), "test examples")


## 1. Load the base model (4-bit, via Unsloth)

`unsloth/Phi-4-mini-instruct-bnb-4bit` is Unsloth's pre-quantized build of
Phi-4-mini — using their pre-quantized version (rather than quantizing a
full-precision checkpoint yourself) is what keeps this fast and light
enough for a free T4. Swap the model name for
`unsloth/Qwen2.5-3.5B-Instruct-bnb-4bit` if you'd rather fine-tune Qwen —
check Unsloth's Hugging Face org page for the exact current tag, since
their catalog of pre-quantized models is updated regularly.

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Phi-4-mini-instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)


## 2. Generate BASE-MODEL predictions first

Do this *before* attaching any LoRA adapter, so these numbers are a clean
"zero-shot, untouched model" baseline -- this is the "before" half of your
before/after comparison.

In [ ]:
import json
from prompt_template import build_prompt

FastLanguageModel.for_inference(model)

test_examples = [json.loads(l) for l in open("data/test.jsonl")]

os_makedirs = __import__("os").makedirs
os_makedirs("results", exist_ok=True)

with open("results/predictions_base.jsonl", "w") as f:
    for i, ex in enumerate(test_examples):
        prompt = build_prompt(ex["input_text"])
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
        full = tokenizer.decode(out[0], skip_special_tokens=True)
        gen_only = full[len(tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)):].strip()
        f.write(json.dumps({"raw_output": gen_only}) + "\n")
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(test_examples)}")

print("Saved results/predictions_base.jsonl")


## 3. Attach LoRA and fine-tune

`r=16` and the target modules below are Unsloth's standard defaults for
this model family -- a reasonable starting point, not something you need to
tune extensively for a project this size.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


In [ ]:
from datasets import Dataset
from prompt_template import build_training_example

train_examples = [json.loads(l) for l in open("data/train.jsonl")]

def to_text(ex):
    target_json = json.dumps(ex["fields"])
    return {"text": build_training_example(ex["input_text"], target_json) + tokenizer.eos_token}

train_dataset = Dataset.from_list(train_examples).map(to_text)
print(train_dataset[0]["text"])


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not __import__("torch").cuda.is_bf16_supported(),
        bf16=__import__("torch").cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs/checkpoints",
        report_to="none",
    ),
)

trainer_stats = trainer.train()


In [ ]:
model.save_pretrained("outputs/lora_adapter")
tokenizer.save_pretrained("outputs/lora_adapter")
print("Saved adapter to outputs/lora_adapter")


## 4. Generate FINE-TUNED predictions on the same test set

In [ ]:
FastLanguageModel.for_inference(model)

with open("results/predictions_finetuned.jsonl", "w") as f:
    for i, ex in enumerate(test_examples):
        prompt = build_prompt(ex["input_text"])
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
        full = tokenizer.decode(out[0], skip_special_tokens=True)
        gen_only = full[len(tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)):].strip()
        f.write(json.dumps({"raw_output": gen_only}) + "\n")
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(test_examples)}")

print("Saved results/predictions_finetuned.jsonl")


## 5. Score both and see the actual before/after numbers

This step needs no GPU -- it's the same `evaluate.py` you can re-run later
on your own laptop straight from the saved prediction files, which is
exactly the point: your evaluation is reproducible by anyone without a GPU.

In [ ]:
!python src/evaluate.py --test data/test.jsonl \
    --predictions results/predictions_base.jsonl \
    --label "Base model (zero-shot)" \
    --save_json results/metrics_base.json

!python src/evaluate.py --test data/test.jsonl \
    --predictions results/predictions_finetuned.jsonl \
    --label "Fine-tuned (LoRA)" \
    --save_json results/metrics_finetuned.json \
    --compare_to results/metrics_base.json


## 6. Get your results back off the Colab VM

Push to your repo (recommended -- keeps everything, including the adapter
weights if they're small enough, or just the results/ and metrics under
git), or download individual files via Colab's file browser.

In [ ]:
!git config user.email "you@example.com"
!git config user.name "Your Name"
!git add results/ outputs/lora_adapter/adapter_config.json outputs/lora_adapter/adapter_model.safetensors
!git commit -m "Add fine-tuning results and LoRA adapter"
# !git push   # uncomment once you've set up credentials (a GitHub PAT, or push manually from your laptop instead)
